# B · Compare host feature profiles by pathogen group

This notebook compares the **resolved question sets** for every configured host and pathogen group. Each matrix cell is the Jaccard similarity: shared applicable features divided by the union of applicable features.

The interactive section compares two hosts and a pathogen group using only their shared feature questions. Each pathogen is one row and each shared feature is one input column. It uses IMSTE when at least three scored pathogen rows are available, with a PCA fallback for two rows. Circle, square, and diamond markers distinguish Both, First, and Second host associations.

In [ ]:
import sys
from pathlib import Path

from IPython.display import display

# Find the version4 project root when launched from either the repo or notebooks directory.
candidates = [Path.cwd().resolve(), Path.cwd().resolve() / "notebooks"]
PROJECT_ROOT = next(
    (parent for candidate in candidates for parent in (candidate, *candidate.parents)
     if (parent / "configs/host_choices.yaml").exists()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the version4 project root (configs/host_choices.yaml).")

PACKAGE_ROOT = PROJECT_ROOT
NOTEBOOK_SRC = PROJECT_ROOT / "notebooks/src"
for import_path in (PACKAGE_ROOT, NOTEBOOK_SRC):
    if str(import_path) not in sys.path:
        sys.path.insert(0, str(import_path))

from host_feature_comparison import (
    build_jaccard_analysis,
    create_interactive_host_comparison,
    plot_jaccard_matrices,
)

# Set to a collection such as {"pathogen_host_pair"} to restrict the comparison.
SCOPE_FILTER = None
analysis = build_jaccard_analysis(PROJECT_ROOT, scope_filter=SCOPE_FILTER)

## 1 · Resolved profile sizes

Host aliases that share an NCBI taxon ID are collapsed to one profile. Feature identities include the feature ID, question version, and scope.

In [ ]:
display(analysis["profile_counts"].sort_values(["pathogen_group", "host"]).reset_index(drop=True))

## 2 · Jaccard matrices

Rows and columns use the same configured host order. Empty profiles produce undefined (`NaN`) comparisons; the diagonal is 1 for non-empty profiles.

In [ ]:
for pathogen_group, matrix in analysis["matrices"].items():
    print(f"{pathogen_group} · host feature-set Jaccard matrix")
    display(matrix)

In [ ]:
plot_jaccard_matrices(analysis)

## 3 · Pairwise overlap details

The table exposes the shared-feature count and union size behind each matrix score.

In [ ]:
display(
    analysis["pairwise_comparisons"]
    .sort_values(["pathogen_group", "jaccard", "host_a", "host_b"], ascending=[True, False, True, True])
    .reset_index(drop=True)
)

## 4 · Compare two hosts in one shared-feature space

The map uses the intersection of the selected hosts’ resolved feature identities. Each dot is a pathogen; each shared feature is an input column. A pathogen is included when either host has a confirmed association and scored values for the shared features. Missing feature values are never replaced by zero. The map reports how many confirmed pathogens have the feature scores needed to plot. Hover points to inspect each host’s association status and supporting PMID counts.

In [ ]:
display(create_interactive_host_comparison(PROJECT_ROOT, scope_filter=SCOPE_FILTER))